# Sprint 4: Three-settlement extension (Mathare + Mukuru)

Proposal weeks 7–8: retrain / fine-tune **both** model heads on Kibera + Mathare + Mukuru and report whether the validation bars are met.

| Head | Metric | Target |
|------|--------|--------|
| Attention U-Net (flood) | IoU (transfer bar) / F1 (NFR) | ≥ 0.60 / ≥ 0.75 |
| GraphSAGE (deprivation) | macro-F1 | ≥ 0.70 (`config.GRAPHSAGE_FINAL_MACRO_F1_TARGET`) |

**What this notebook does**

1. Checks out the Sprint 4 branch (or `main` once merged).
2. Rebuilds proximity graphs for all three settlements (Overpass mirror retries).
3. Loads tiled flood stacks when present on Drive; scaffolds the path otherwise.
4. Jointly trains GraphSAGE on the combined three-settlement graph (per-settlement stratified split).
5. Trains / fine-tunes the Attention U-Net on pooled tiles with per-settlement spatial holdout (no cross-settlement location leakage).
6. Prints per-settlement + pooled metrics and whether thresholds are met.

**Offline / Cloud Agent note.** Real Mathare/Mukuru GEE exports and Overpass pulls only run here in Colab. The `src/models/multi_settlement.py` helpers are covered by offline unit tests with synthetic data.


## 0. Before you run

- **Runtime → Change runtime type → T4 GPU** recommended (U-Net fine-tune benefits; GraphSAGE is fine on CPU).
- Sprint 1 tiling for each settlement should produce `data/tiles/<settlement>/manifest_train.json` on Drive (or rebuild graphs-only first and tile later).
- Set `GEE_PROJECT_ID` if you will kick off missing raster exports.
- Mount Drive so graphs, tiles, and checkpoints persist across Colab disconnects.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — GraphSAGE is fine on CPU; U-Net will be slower.")


In [ ]:
%%bash
# Prefer the Sprint 4 PR branch while open; switch BRANCH to main after merge.
BRANCH=cursor/sprint4-three-settlements-cc98
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git

if [ -d /content/nairobi-risk-mapping/.git ]; then
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || git fetch origin main
  git checkout "$BRANCH" 2>/dev/null || git checkout main
  git pull --ff-only origin "$BRANCH" 2>/dev/null || git pull --ff-only origin main
else
  git clone "$REPO" /content/nairobi-risk-mapping
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || true
  git checkout "$BRANCH" 2>/dev/null || git checkout main
fi
cd /content/nairobi-risk-mapping
git rev-parse --abbrev-ref HEAD
ls src/models/multi_settlement.py


In [ ]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt


In [ ]:
# --- config knobs ---
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    # Persist heavy artefacts on Drive (Colab local disk is wiped on disconnect).
    DRIVE_ROOT = Path("/content/drive/MyDrive/nairobi_risk_mapping")
except ImportError:
    print("Not in Colab — artefacts stay under the repo working directory.")
    DRIVE_ROOT = Path(".")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Symlink repo data/ -> Drive so DataPipeline paths stay relative.
DATA_LINK = Path("data")
if not (DATA_LINK.is_symlink() or DATA_LINK.exists()):
    target = DRIVE_ROOT / "data"
    target.mkdir(parents=True, exist_ok=True)
    DATA_LINK.symlink_to(target)

SETTLEMENT_KEYS = ("kibera", "mathare", "mukuru")
REBUILD_GRAPHS = False          # True forces fresh OSM/Overpass pulls
INCLUDE_RASTER_EXPORT = False  # True kicks off async GEE exports (slow)
FINE_TUNE_UNET_FROM = None     # e.g. "checkpoints/kibera_attention_unet_best.pt"
GEE_PROJECT_ID = None          # or os.environ / your GCP project id
GRAPHSAGE_EPOCHS = 100
UNET_EPOCHS = 40
CHECKPOINT_DIR = str(DRIVE_ROOT / "checkpoints" / "sprint4")
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print("DATA ->", DATA_LINK.resolve())
print("CHECKPOINTS ->", CHECKPOINT_DIR)

gs_threshold = None
unet_threshold = None
unet_result = None


## 1. Repo check — Sprint 4 APIs exist


In [ ]:
from src import config
from src.models.multi_settlement import (
    combine_settlement_graphs,
    evaluate_flood_entries_by_settlement,
    evaluate_graphsage_by_settlement,
    flood_threshold_report,
    graphsage_threshold_report,
    load_multi_settlement_manifests,
    load_settlement_graphs,
    multi_settlement_spatial_holdout,
    prepare_multi_settlement_data,
)
from src.models.graphsage_model import GraphSAGEModel
from src.models.attention_unet_model import AttentionUNetModel
from src.models.dataset import EVENT_S1_LAYOUT, DEFAULT_LAYOUT

assert config.GRAPHSAGE_FINAL_MACRO_F1_TARGET == 0.70
assert config.ATTENTION_UNET_TRANSFER_IOU_TARGET == 0.60
assert config.ATTENTION_UNET_FINAL_F1_TARGET == 0.75
assert tuple(config.SETTLEMENT_KEYS) == SETTLEMENT_KEYS
print("Sprint 4 targets OK:", {
    "graphsage_macro_f1": config.GRAPHSAGE_FINAL_MACRO_F1_TARGET,
    "unet_iou": config.ATTENTION_UNET_TRANSFER_IOU_TARGET,
    "unet_f1": config.ATTENTION_UNET_FINAL_F1_TARGET,
})


## 2. Rebuild / load graphs for all three settlements

Uses `DataPipeline.acquire_graph_data()` (Overpass mirror retries from Sprint 3). Set `REBUILD_GRAPHS=True` if Drive graphs predate deprivation labels.


In [ ]:
from src.data_pipeline import DataPipeline
import torch

graphs_dir = Path("data/graphs")
graphs_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_GRAPHS or not all((graphs_dir / f"{k}_graph.pt").exists() for k in SETTLEMENT_KEYS):
    for key in SETTLEMENT_KEYS:
        print("=" * 60, key)
        pipe = DataPipeline(settlement_key=key, gee_project_id=GEE_PROJECT_ID)
        if INCLUDE_RASTER_EXPORT:
            task = pipe.acquire_raster_stack()
            print("GEE export task:", task)
        data, integrity = pipe.acquire_graph_data()
        print(key, "nodes=", data.num_nodes, "integrity=", integrity)
else:
    print("Using existing graphs in", graphs_dir.resolve())

graphs = load_settlement_graphs(graphs_dir, settlement_keys=SETTLEMENT_KEYS)
print("Loaded settlements:", list(graphs))
for k, g in graphs.items():
    print(f"  {k}: nodes={g.num_nodes} y_unique={torch.unique(g.y).tolist()}")


## 3. GraphSAGE — joint three-settlement training

Graphs are combined as a **disjoint union** (no cross-settlement edges). Splits are stratified **within each settlement** (60/20/20). Selection metric: validation macro-F1.


In [ ]:
if len(graphs) < 1:
    raise RuntimeError("No graphs loaded — rebuild with REBUILD_GRAPHS=True.")

combined = prepare_multi_settlement_data(graphs, seed=42)
print("Combined nodes:", combined.num_nodes)
print("Split sizes:",
      int(combined.train_mask.sum()),
      int(combined.val_mask.sum()),
      int(combined.test_mask.sum()))

gs_model = GraphSAGEModel(
    in_channels=combined.x.size(1),
    hidden_channels=64,
    num_classes=config.DEPRIVATION_NUM_CLASSES,
    dropout=0.5,
)
gs_result = gs_model.fit(
    combined,
    epochs=GRAPHSAGE_EPOCHS,
    learning_rate=5e-3,
    weight_decay=5e-4,
    checkpoint_dir=str(Path(CHECKPOINT_DIR) / "graphsage"),
    early_stopping_patience=25,
    seed=42,
)
print("best_val_macro_f1:", gs_result["best_val_macro_f1"])
print("meets_final_target (pooled val):", gs_result["meets_final_target"],
      "target=", gs_result["final_target"])


In [ ]:
# Per-settlement + pooled reports on val and test
for split in ("val", "test"):
    report = gs_model.evaluate_by_settlement(combined, split=split)
    print(f"\n=== GraphSAGE {split} ===")
    for key, r in report["per_settlement"].items():
        print(f"  {key:8s}  macro_f1={r['macro_f1']:.4f}  acc={r['accuracy']:.4f}")
    print(f"  {'pooled':8s}  macro_f1={report['pooled']['macro_f1']:.4f}")
    thr = graphsage_threshold_report(
        {k: v["macro_f1"] for k, v in report["per_settlement"].items()},
        report["pooled"]["macro_f1"],
    )
    print("  threshold:", {k: thr[k] for k in (
        "meets_pooled_target", "all_settlements_meet_target", "target",
    )})
    if split == "val":
        gs_threshold = thr


## 4. Attention U-Net — pooled tiles + per-settlement spatial holdout

Requires Sprint 1 tiled stacks on Drive (`data/tiles/<settlement>/manifest_train.json`). If Mathare/Mukuru tiles are missing, this section skips with a clear message — rebuild via GEE export + `tile_raster_stack` in notebook 01 / 02 v2 first.

Holdouts are **per settlement**: augmented variants of the same `(row_off, col_off)` stay together; the same numeric offsets in different settlements are independent.


In [ ]:
import numpy as np

tiles_root = Path("data/tiles")
manifests = load_multi_settlement_manifests(
    tiles_root, settlement_keys=SETTLEMENT_KEYS, split="train", require_all=False,
)
print("Tile manifests found:", {k: len(v) for k, v in manifests.items()})

unet_threshold = None
unet_result = None

if len(manifests) < 2:
    print(
        "SKIPPING U-Net training: need tile manifests for at least two settlements. "
        "Run Sprint 1 tiling for Mathare/Mukuru in Colab (GEE → Drive → tile_raster_stack), "
        "then re-run this section. GraphSAGE above does not depend on tiles."
    )
else:
    split = multi_settlement_spatial_holdout(manifests, holdout_fraction=0.25, seed=42)
    print("Holdout locations:", split["holdout_locations"])
    print("Train/val entries:", len(split["train"]), len(split["val"]))

    # Prefer EVENT_S1_LAYOUT when tile band count matches (Sprint 2 v2 stacks).
    sample = np.load(split["train"][0]["path"])
    layout = EVENT_S1_LAYOUT if sample.shape[0] == len(EVENT_S1_LAYOUT.band_names) else DEFAULT_LAYOUT
    print("Using layout:", layout.band_names, "in_channels=", layout.in_channels)

    unet = AttentionUNetModel(
        layout=layout,
        dropout=0.3,
        pretrained_checkpoint=None,  # set Sen1Floods11 path if available
        freeze_pretrained_encoder=False,
    )
    if FINE_TUNE_UNET_FROM:
        unet.load_checkpoint(FINE_TUNE_UNET_FROM)
        print("Fine-tuning from", FINE_TUNE_UNET_FROM)

    unet_result = unet.fit(
        split["train"],
        split["val"],
        epochs=UNET_EPOCHS,
        batch_size=4,
        learning_rate=1e-3,
        weight_decay=1e-4,
        early_stopping_patience=12,
        checkpoint_dir=str(Path(CHECKPOINT_DIR) / "attention_unet"),
    )
    print("best_val_iou:", unet_result["best_val_iou"])

    # Reload best checkpoint for settlement-wise summary on the val set.
    best_ckpt = Path(CHECKPOINT_DIR) / "attention_unet" / "attention_unet_best.pt"
    if best_ckpt.exists():
        unet.load_checkpoint(str(best_ckpt))

    flood_eval = evaluate_flood_entries_by_settlement(unet, split["val"])
    print("\n=== U-Net val (per settlement + pooled) ===")
    for key, r in flood_eval["per_settlement"].items():
        print(f"  {key:8s}  iou={r['iou']:.4f}  f1={r['f1']:.4f}")
    print(f"  {'pooled':8s}  iou={flood_eval['pooled']['iou']:.4f}  "
          f"f1={flood_eval['pooled']['f1']:.4f}")

    unet_threshold = flood_threshold_report(
        {k: {"iou": v["iou"], "f1": v["f1"]} for k, v in flood_eval["per_settlement"].items()},
        flood_eval["pooled"]["iou"],
        flood_eval["pooled"]["f1"],
    )
    print("threshold:", {
        k: unet_threshold[k] for k in (
            "meets_pooled_iou_target", "meets_pooled_f1_target",
            "iou_target", "f1_target",
        )
    })


## 5. Threshold summary


In [ ]:
print("=" * 60)
print("SPRINT 4 THRESHOLD SUMMARY")
print("=" * 60)

print("\nGraphSAGE (deprivation) — target macro-F1 >=",
      config.GRAPHSAGE_FINAL_MACRO_F1_TARGET)
if gs_threshold is not None:
    print("  pooled meets target:", gs_threshold["meets_pooled_target"],
          f"(macro_f1={gs_threshold['pooled_macro_f1']:.4f})")
    for k, v in gs_threshold["per_settlement_macro_f1"].items():
        print(f"  {k:8s} macro_f1={v:.4f}  meets={gs_threshold['meets_per_settlement_target'][k]}")
    print("  all settlements meet:", gs_threshold["all_settlements_meet_target"])
else:
    print("  (no GraphSAGE threshold report — training cell did not run)")

print("\nAttention U-Net (flood) — IoU >=",
      config.ATTENTION_UNET_TRANSFER_IOU_TARGET,
      "/ F1 >=", config.ATTENTION_UNET_FINAL_F1_TARGET)
if unet_threshold is not None:
    print("  pooled IoU meets:", unet_threshold["meets_pooled_iou_target"],
          f"(iou={unet_threshold['pooled_iou']:.4f})")
    print("  pooled F1 meets:", unet_threshold["meets_pooled_f1_target"],
          f"(f1={unet_threshold['pooled_f1']:.4f})")
    for k in unet_threshold["per_settlement_iou"]:
        print(f"  {k:8s} iou={unet_threshold['per_settlement_iou'][k]:.4f}  "
              f"f1={unet_threshold['per_settlement_f1'][k]:.4f}")
else:
    print("  (U-Net skipped — tile manifests missing for Mathare/Mukuru; run tiling in Colab)")

print("\nCheckpoints under:", CHECKPOINT_DIR)
print("Next: Sprint 5 late-fusion policy engine (do not start here).")


## Done

- GraphSAGE joint training path is fully wired offline + in this notebook.
- U-Net multi-settlement path needs real Mathare/Mukuru tiles from GEE (Colab-only).
- Do **not** treat synthetic Cloud-Agent metrics as real accuracy — re-run this notebook in Colab for reportable numbers.
